In [18]:
!pip install -q --no-cache-dir google-generativeai pydantic json-repair pymupdf Pillow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.3/51.3 kB 245.1 MB/s eta 0:00:00


In [19]:
# ============================================================
# CELL 1 — INSTALL GEMINI & EXTRACTION DEPENDENCIES
# ============================================================
!pip install -q --no-cache-dir \
    "google-genai>=0.1.0" \
    "pydantic>=2.0.0" \
    "json-repair" \
    "pymupdf>=1.24.0" \
    "Pillow"

print("✅ Dependencies installed successfully for Gemini Vision Pipeline!")

✅ Dependencies installed successfully for Gemini Vision Pipeline!


In [ ]:
# ============================================================
# CELL 2 — SETUP GEMINI API CLIENT
# ============================================================
import os
from google import genai


GEMINI_API_KEY = "AQ.Ab8RN6J0Dphy_bRfVE1MSuosATvxv9WUqa94N-rZ9AYcSlYTyQ"

# if GEMINI_API_KEY =="AQ.Ab8RN6L-mezrySY4iV9f6NtLxPPDGVI0Amjzhhgh-p_3ycZ6ow":
#     print("⚠️ WARNING: Please replace 'YOUR_GEMINI_API_KEY_HERE' with your actual key from AI Studio!")

client = genai.Client(api_key=GEMINI_API_KEY)

print("✅ Gemini API Client Initialized Successfully!")

✅ Gemini API Client Initialized Successfully!


In [ ]:
# ============================================================
# CELL 3 — UNIVERSAL DOCUMENT LOADER (PDF & IMAGES)
# ============================================================
import os
import pymupdf
from PIL import Image

def load_document_as_images(file_path: str, dpi: int = 200):
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Document not found at path: {file_path}")

    ext = os.path.splitext(file_path)[1].lower()
    images = []

    print("=" * 70)
    print(f"PROCESSING DOCUMENT: {os.path.basename(file_path)}")
    print("=" * 70)

    if ext == ".pdf":
        doc = pymupdf.open(file_path)
        for page_idx in range(len(doc)):
            page = doc[page_idx]
            pix = page.get_pixmap(dpi=dpi, alpha=False)
            img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
            images.append(img)
            print(f"  --> Extracted Page {page_idx + 1}")
        doc.close()
    elif ext in [".png", ".jpg", ".jpeg", ".webp"]:
        img = Image.open(file_path).convert("RGB")
        images.append(img)
        print("  --> Image Loaded Successfully")
    else:
        raise ValueError("Unsupported file extension.")
        
    return images


DOCUMENT_PATH = "/kaggle/input/datasets/nataliemmoris/case-7/bumped-into-a-truck-and-im-freaking-out-v0-dr3alovmx8kf1.jpg"

if os.path.exists(DOCUMENT_PATH):
    document_images = load_document_as_images(DOCUMENT_PATH, dpi=200)
else:
    print(f"⚠️ Warning: File {DOCUMENT_PATH} not found. Please upload it.")

PROCESSING DOCUMENT: bumped-into-a-truck-and-im-freaking-out-v0-dr3alovmx8kf1.jpg
  --> Image Loaded Successfully


In [22]:
# ============================================================
# CELL 4 — DATA SCHEMAS & PROMPT DEFINITION
# ============================================================
import json
from typing import List, Optional, Literal
from pydantic import BaseModel, Field, field_validator, model_validator

# ------------------------------------------------------------
# 1. SHARED SYSTEM TAXONOMY
# ------------------------------------------------------------
VehicleType = Literal[
    "PASSENGER CAR", "TRUCK", "MULTIPURPOSE PASSENGER VEHICLE (MPV)",
    "BUS", "INCOMPLETE VEHICLE", "OTHER"
]

BodyClass = Literal[
    "Cargo Van", "Convertible/Cabriolet", "Coupe", "Crossover Utility Vehicle (CUV)",
    "Hatchback/Liftback/Notchback", "Incomplete", "Incomplete - Chassis Cab (Number of Cab Unknown)",
    "Incomplete - Chassis Cab (Single Cab)", "Incomplete - Cutaway", "Incomplete - Motor Home Chassis",
    "Incomplete - Stripped Chassis", "Minivan", "Pickup", "Roadster", "Sedan/Saloon",
    "Sport Utility Truck (SUT)", "Sport Utility Vehicle (SUV)/Multi-Purpose Vehicle (MPV)",
    "Truck", "Van", "Wagon", "Other"
]

CarPart = Literal[
    "Bumper", "Front Bumper", "Rear Bumper", "Bumper Reinforcement", "Bumper Support",
    "Bumper Fascia", "Lower Valance", "Hood", "Grille", "Headlight", "Taillight",
    "Fender", "Quarter Panel","Front Door","Rear Door", "Door", "Door Handle", "Door Jamb", "Side Mirror",
    "Windshield", "Side Window", "Rear Window", "Windshield Header", "Rear Window Header",
    "Rear Window Ledge", "Roof", "Roof Rail", "Roof Header", "Roof Rack", "Headliner",
    "Rocker Panel", "Molding", "Weatherstrip", "A-Pillar", "B-Pillar", "C-Pillar", "D-Pillar", "Other Pillar",
    "Trunk Lid", "Rear Hatch", "Tailgate", "Spoiler", "Body Trim", "Running Board",
    "Truck Cab", "Pickup Bed", "Bedside Panel", "Bed Rail", "Vehicle Canopy",
    "Wheel", "Tire", "Rim", "Wheel Cover", "Wheel Hub", "Wheel Well", "Axle",
    "Suspension", "Control Arm", "Differential", "Drivetrain", "Frame", "Frame Rail",
    "Crossmember", "Radiator", "Radiator Support", "Radiator Mounting Bracket",
    "Skid Plate", "Underbody", "Seat", "License Plate", "Trailer Hitch","Park Sensor", "Sensor Cap", "Radar", "Other"
]

OperationType = Literal["Replace", "Repair","Refinish", "Blend", "Not Specified", "Exclude"]

ALLOWED_CAR_PARTS = set(CarPart.__args__)

# ------------------------------------------------------------
# 2. SCHEMAS
# ------------------------------------------------------------
class ExtractedCarInfo(BaseModel):
    car_make: Optional[str] = Field(None, description="Vehicle Make (e.g. Toyota, Dodge).")
    car_model: Optional[str] = Field(None, description="Vehicle Model (e.g. Camry, Ram).")
    car_model_year: Optional[int] = Field(None, description="4-digit Model Year.")
    vehicle_type: VehicleType = Field("PASSENGER CAR", description="Mapped Vehicle Type enum.")
    body_class: BodyClass = Field("Sedan/Saloon", description="Mapped Body Class enum.")

class ExtractedRepairedPart(BaseModel):
    original_description: str = Field(..., description="EXACT line item text printed on the invoice.")
    standard_part: CarPart = Field(..., description="Mapped part corresponding directly to system CarPart taxonomy.")
    operation: OperationType = Field(..., description="Action code: Replace, Repair, or Blend.")

    @field_validator('operation', mode='before')
    @classmethod
    def parse_and_validate_operation(cls, v: str) -> str:
        v_clean = str(v).strip().lower()
        if any(x in v_clean for x in ["r&i", "r/i", "rbi", "rai", "remove"]):
            return "Exclude"
        if any(x in v_clean for x in ["repl", "replace", "repi", "rep/l"]):
            return "Replace"
        elif any(x in v_clean for x in ["blnd", "blend", "bind"]):
            return "Blend"
        elif any(x in v_clean for x in ["rpr", "repair"]):
            return "Repair"
        else:
            return "Exclude"

    @field_validator('standard_part', mode='before')
    @classmethod
    def auto_fix_part_typos(cls, v: str) -> str:
        if not isinstance(v, str) or not v.strip():
            return "Other"
        
        v_clean = v.strip()
        typo_map = {
            "tailight": "Taillight", "tail light": "Taillight",
            "head light": "Headlight", "front bumper": "Front Bumper",
            "rear bumper": "Rear Bumper", "door shell": "Door",
            "rocker panel": "Rocker Panel", "quarter panel": "Quarter Panel",
            "rocker molding": "Rocker Panel", "stone guard": "Body Trim",
            "door w/strip": "Door Jamb", "belt molding": "Body Trim",

            "auto park sensor": "Park Sensor", 
            "park sensor": "Park Sensor", 
            "reverse sensor": "Park Sensor",
            "reverse sensor cap": "Sensor Cap",
            "sensor cap": "Sensor Cap"
        }
        
        if v_clean.lower() in typo_map:
            return typo_map[v_clean.lower()]
            
        formatted = v_clean.title()
        if formatted in ALLOWED_CAR_PARTS:
            return formatted
        return "Other"

class CleanInvoiceExtraction(BaseModel):
    car_info: ExtractedCarInfo = Field(..., description="Header vehicle meta information extracted from the invoice.")
    repaired_parts: List[ExtractedRepairedPart] = Field(default_factory=list, description="List of physical repaired/replaced/blended parts.")

    @model_validator(mode='after')
    def strict_labor_and_supplies_filter(self):
        filtered_list = []
        section_headers = ["rear door", "front door", "quarter panel", "rear lamps", "pillars", "rear bumper", "pillars, rocker & floor"]
        blacklisted_terms = [
            "coat", "spray", "mask", "sand", "reset", "scan", "waste", "hazardous",
            "calibrate", "labor", "buff", "undercoat", "deadner", "oil", "fluid",
            "washer", "cleaner", "coolant", "spark plug", "grease", "chemical",
            "zip tie", "filter", "shim", "seal kit", "syn", "synthetic", "drum",
            "75w", "5w40", "dot4", "r&i", "r/i", "subl", "pre and post scan", "clear coat"
        ]
        seen_descriptions = set()

        for part in self.repaired_parts:
            desc_clean = part.original_description.strip().lower()

            if part.operation == "Exclude":
                continue
            if desc_clean in section_headers:
                continue
            if any(term in desc_clean for term in blacklisted_terms):
                continue
            if desc_clean in seen_descriptions:
                continue
            
            seen_descriptions.add(desc_clean)
            filtered_list.append(part)
            
        self.repaired_parts = filtered_list
        return self

# ------------------------------------------------------------
# 3. PROMPT GENERATION
# ------------------------------------------------------------
SCHEMA_STR = json.dumps(CleanInvoiceExtraction.model_json_schema(), indent=2)

SMART_INVOICE_PROMPT = f"""
You are a Spatial OCR Document Engine for Automotive Estimates.

CRITICAL EXECUTION LOGIC (STRICT STEP-BY-STEP AUDITING):

STEP 1: IDENTIFY TABLE COLUMNS
Locate the table header: [Line] [Oper] [Description] ...
The 'Oper' column is strictly positioned between [Line] and [Description].

STEP 2: ROW-BY-ROW COLUMN AUDIT (DO NOT SKIP ANY CHARACTERS):
For EVERY numbered row, extract the EXACT raw text inside the 'Oper' column bounds.
Apply the following strict filters:

A. EMPTY/BLANK Oper Cell:
   - If the cell under 'Oper' is empty, space, or missing (e.g., "O/H bumper assy"): DISCARD ROW IMMEDIATELY. Do not invent an operation.

B. ALLOWED OPER VALUES (Exact Match):
   - 'Repl' or 'Repi' -> Map operation to "Replace"
   - 'Rpr' or 'Repair' -> Map operation to "Repair"
   - 'Blnd' or 'Bind' -> Map operation to "Blend"
   - 'Refn' or 'Paint' -> Map operation to "Refinish"

C. STRICT DISCARD VALUES:
   - 'R&I', 'R/I', 'RBI', 'RAI', 'Refn', 'Subl' -> DISCARD ROW IMMEDIATELY.

STEP 3: CATEGORY & DESCRIPTION CLEANUP
- Ignore section header rows (e.g., REAR BUMPER, VEHICLE DIAGNOSTICS).
- Ignore non-physical items/scans/fees (Pre-repair scan, Post-repair scan, Clear coat).
-- Extract physical sensors (e.g., Reverse sensor, Park sensor, Radar) as valid parts under 'Park Sensor' or 'Sensor Cap'.
Return ONLY valid raw JSON matching {SCHEMA_STR}. Do not include any extra introductory text.
- Do NOT discard body parts that are being refinished or blended (e.g. "Blend rt. roof rail" is VALID).
- IGNORE paint materials/supplies and fees (e.g., Pre-repair scan, Post-repair scan, Clear coat, Flex additive, Hazardous waste).
- Extract physical sensors (e.g., Reverse sensor, Park sensor, Radar) as valid parts under 'Park Sensor' or 'Sensor Cap'.
Return ONLY valid raw JSON matching {SCHEMA_STR}. Do not include any extra introductory text.
"""

print("✅ Schemas & Prompt compiled successfully!")

✅ Schemas & Prompt compiled successfully!


In [145]:
# ============================================================
# CELL 5 — GEMINI EXTRACTION ENGINE & EXECUTION
# ============================================================
import re
import json
from google.genai import types
from json_repair import repair_json

def extract_json(raw_text: str) -> dict:
    match = re.search(r'```(?:json)?\s*([\s\S]*?)\s*```', raw_text)
    text = match.group(1).strip() if match else raw_text.strip()
    
    try:
        return json.loads(text)
    except Exception:
        pass

    try:
        repaired_string = repair_json(text)
        return json.loads(repaired_string)
    except Exception:
        cleaned = re.sub(r',\s*([\}\]])', r'\1', text)
        cleaned = re.sub(r"(?<!\\)'", '"', cleaned)
        return json.loads(cleaned)

def process_invoice_document_gemini(images, prompt=SMART_INVOICE_PROMPT) -> CleanInvoiceExtraction:
    if not images:
        raise ValueError("No document images provided.")

    contents = []
    for img in images:
        contents.append(img)
    contents.append(prompt)

    print("🚀 RUNNING GEMINI 3.6 FLASH VISION INFERENCE (FREE)...")

    response = client.models.generate_content(
        model='gemini-3.6-flash',
        contents=contents,
        config=types.GenerateContentConfig(
            temperature=0.0,
            response_mime_type="application/json"
        )
    )

    output_text = response.text
    raw_json = extract_json(output_text)
    validated_output = CleanInvoiceExtraction(**raw_json)
    return validated_output

# ------------------------------------------------------------
# EXECUTION PIPELINE
# ------------------------------------------------------------
if __name__ == "__main__":
    if "document_images" in globals() and document_images:
        result: CleanInvoiceExtraction = process_invoice_document_gemini(document_images)
        print("\n✅ EXTRACTION SUCCESSFUL VIA GEMINI (FREE):")
        print(json.dumps(result.model_dump(), indent=2, ensure_ascii=False))

🚀 RUNNING GEMINI 3.6 FLASH VISION INFERENCE (FREE)...

✅ EXTRACTION SUCCESSFUL VIA GEMINI (FREE):
{
  "car_info": {
    "car_make": "Dodge",
    "car_model": "1500",
    "car_model_year": 2014,
    "vehicle_type": "TRUCK",
    "body_class": "Pickup"
  },
  "repaired_parts": [
    {
      "original_description": "Add for park sensor",
      "standard_part": "Park Sensor",
      "operation": "Replace"
    },
    {
      "original_description": "Bumper primed, w/park sensor w/dual exh",
      "standard_part": "Rear Bumper",
      "operation": "Repair"
    },
    {
      "original_description": "Step pad",
      "standard_part": "Rear Bumper",
      "operation": "Replace"
    }
  ]
}
